# Урок 13 - Пам’ять агента


## Налаштування

Ця нотатка демонструє, як створити агента для бронювання подорожей з **постійною пам’яттю** за допомогою **Microsoft Agent Framework** (MAF).

Ви дізнаєтесь, як різні типи пам’яті агента — робоча, короткочасна та довготривала — впливають на те, як агент зберігає та використовує інформацію під час розмов.

**Вимоги:**
- Проєкт Azure AI Foundry з розгорнутою чат-моделлю (наприклад, `gpt-4o-mini`).
- Увійшли в систему через Azure CLI — виконайте команду `az login` у вашому терміналі.
- `AZURE_AI_PROJECT_ENDPOINT` — ваш кінцевий пункт проєкту Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — назва вашої розгорнутої моделі.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Типи пам’яті агента

Агент штучного інтелекту може використовувати різні види пам’яті, кожен із яких виконує окрему функцію:

### Оперативна пам’ять
Сам потік розмови — повідомлення, обміняні в одній сесії. Агент може повернутися до ранніх повідомлень у тому ж потоці, щоб підтримувати зв’язність. У MAF це створюється через **`agent.create_session()`**, що повертає `AgentSession`.

### Короткочасна пам’ять
Інформація, яка зберігається протягом виконання завдання або сесії, але не зберігається постійно. Наприклад, агент може накопичувати факти під час багатокрокової розмови про планування та використовувати їх для створення остаточного маршруту.

### Довготривала пам’ять
Вподобання та факти, які зберігаються **між сесіями**. Користувачу, що повертається, не потрібно повторювати свої дієтичні обмеження чи стиль подорожей. Довготривала пам’ять зазвичай підтримується зовнішнім сховищем — базою даних, файлом або векторним індексом — і передається агенту через інструменти.


## Робоча пам'ять із сесіями

Найпростіша форма пам'яті — це сесія розмови. Коли ви передаєте ту ж сесію (створену через `agent.create_session()`) до послідовних викликів `agent.run()`, агент бачить повну історію цієї розмови і може пригадати попередні деталі.

Давайте створимо туристичного агента і продемонструємо робочу пам'ять.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агент правильно нагадав бюджет, тому що обидва повідомлення мають одну сесію. Це **робоча пам’ять** — вона існує лише протягом життя сесії.

### Що відбувається з новою гілкою?

Якщо ми створимо **нову** сесію, агент не пам’ятає попередню розмову:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Шаблон довготривалої пам’яті

Щоб запам’ятовувати вподобання користувача **поміж сесіями**, потрібне постійне сховище, що існує поза потоком розмови. Аґент отримує доступ до цього сховища через **інструменти** — функції, які він може викликати для збереження та отримання інформації.

Нижче ми реалізуємо просте сховище вподобань у пам’яті (в продуктивному середовищі ви б використали базу даних або векторний індекс) і відкриємо його як інструменти, які аґент може використовувати.

### Архітектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарій 1 — Користувач, який бронює поїздку на річницю вперше

Сара відвідує вперше. Агент повинен зберегти її вподобання за допомогою інструментів і використовувати їх для рекомендації готелів.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарій 2 — Сара повертається через кілька тижнів

Сара починає **нову розмову** (імітація нової сесії). Робоча пам’ять порожня, але сховище довгострокових уподобань все ще містить її інформацію. Агент повинен витягти її та використати для персоналізації рекомендацій.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Підсумок

У цьому уроці ви дізналися про три типи пам’яті агента та як їх реалізувати за допомогою Microsoft Agent Framework:

| Тип пам’яті | Механізм MAF | Тривалість |
|---|---|---|
| **Робоча** | `agent.create_session()` | Одна розмова |
| **Короткочасна** | Акумульований контекст у межах потоку | Одне завдання / сесія |
| **Довготривала** | Зовнішнє сховище, до якого звертаються через функції `@tool` | Між сесіями |

### Основні висновки
1. **`agent.create_session()`** забезпечує робочу пам’ять — агент бачить всю історію розмови в межах сесії.
2. **Нові сесії втрачають контекст** — без довготривалої пам’яті агент не може пригадати минулі розмови.
3. **Функції `@tool`** долають цю прогалину — вони дозволяють агенту зберігати і отримувати інформацію з постійного сховища.
4. **Персоналізація покращується з часом** — чим більше збережено переваг, тим якісніші рекомендації агента.

### Реальні застосування
- **Обслуговування клієнтів**: Запам’ятовувати історію та переваги клієнта
- **Персональні помічники**: Підтримувати контекст протягом днів або тижнів
- **Охорона здоров’я**: Відстежувати інформацію та уподобання пацієнта
- **Електронна комерція**: Персоналізовані покупки на основі історії

### Наступні кроки
- Замінити словник у пам’яті базою даних або векторним сховищем (наприклад, Azure AI Search)
- Додати термін дії для чутливої до часу інформації
- Створити багатагентні системи з розділеною пам’яттю
- Дослідити нотатник Cognee для пам’яті, підтриманої графом знань


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:  
Цей документ був перекладений із використанням сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Незважаючи на наші зусилля забезпечити точність, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звернутися до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
